In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Análise Final — Respondendo as Perguntas de Negócio do MVP
# MAGIC |---|---|
# MAGIC | **Autor** | João Pedro Paciello |
# MAGIC 1. Qual o impacto da localização (bairro) no preço da diária?
# MAGIC 2. O tipo de imóvel e a capacidade de hóspedes influenciam mais que a localização?
# MAGIC 3. Existe correlação entre avaliação do anúncio (review score) e preço cobrado?
# MAGIC 4. Os preços variam significativamente entre alta e baixa temporada?
# MAGIC 5. Superhosts conseguem cobrar preços acima da média do mercado?

# COMMAND ----------

from pyspark.sql.functions import col, avg, median, count, corr, round as spark_round
import matplotlib.pyplot as plt
import seaborn as sns

CATALOGO = "mvp_airbnb_rj"
SCHEMA_GOLD = "gold"

fato = spark.table(f"{CATALOGO}.{SCHEMA_GOLD}.fato_diarias_airbnb")
dim_localizacao = spark.table(f"{CATALOGO}.{SCHEMA_GOLD}.dim_localizacao")
dim_imovel = spark.table(f"{CATALOGO}.{SCHEMA_GOLD}.dim_imovel")
dim_host = spark.table(f"{CATALOGO}.{SCHEMA_GOLD}.dim_host")
dim_tempo = spark.table(f"{CATALOGO}.{SCHEMA_GOLD}.dim_tempo")

print(f"Total de registros na fato: {fato.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Decisões de tratamento (retomando os achados de Qualidade de Dados)
# MAGIC
# MAGIC 1. **Preço zerado (`flag_price_zero`):** registros com `price = 0` são **excluídos**
# MAGIC    das análises de preço a partir daqui. Um preço de R$ 0,00 não representa uma
# MAGIC    transação real de mercado — é um dado ausente ou quebrado, e mantê-lo distorceria
# MAGIC    qualquer média ou comparação. Os registros continuam preservados na camada Gold
# MAGIC    (não foram apagados do pipeline), apenas filtrados nestas consultas específicas.
# MAGIC 2. **Outliers extremos (`flag_price_outlier`):** **mantidos** nas análises, conforme
# MAGIC    decisão anterior. Para reduzir a distorção que eles causam na média, usamos também
# MAGIC    a **mediana** como medida complementar em todas as agregações de preço — a mediana
# MAGIC    é naturalmente mais robusta a valores extremos que a média. Sempre que os dois
# MAGIC    valores (média x mediana) divergirem muito, isso é um sinal visível da influência
# MAGIC    dos outliers, e comentamos isso explicitamente nas conclusões.

# COMMAND ----------

fato_valido = fato.filter(col("flag_price_zero") == False)
print(f"Registros após excluir price = 0: {fato_valido.count()} (de {fato.count()})")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Pergunta 1 — Qual o impacto da localização (bairro) no preço da diária?

# COMMAND ----------

preco_por_bairro = (
    fato_valido
    .join(dim_localizacao, fato_valido.sk_localizacao == dim_localizacao.sk_localizacao, "inner")
    .groupBy("neighbourhood_cleansed")
    .agg(
        spark_round(avg("price"), 2).alias("preco_medio"),
        spark_round(median("price"), 2).alias("preco_mediano"),
        count("*").alias("qtd_anuncios"),
    )
    .filter(col("qtd_anuncios") >= 100) 
    .orderBy(col("preco_mediano").desc())
)

display(preco_por_bairro.limit(20))

# COMMAND ----------

# MAGIC %md
# MAGIC ### Investigação dos outliers que distorciam o ranking por média
# MAGIC Aqui investigamos os anúncios de maior preço nesses bairros específicos, para entender se são
# MAGIC outliers legítimos (imóveis realmente premium e atípicos) ou possíveis erros de dado.

# COMMAND ----------

bairros_investigar = ["Ramos", "Padre Miguel", "Bangu"]

anuncios_extremos = (
    fato_valido
    .join(dim_localizacao, fato_valido.sk_localizacao == dim_localizacao.sk_localizacao, "inner")
    .join(dim_imovel, fato_valido.sk_imovel == dim_imovel.sk_imovel, "inner")
    .filter(col("neighbourhood_cleansed").isin(bairros_investigar))
    .select(
        col("neighbourhood_cleansed"),
        fato_valido.sk_imovel.alias("sk_imovel"),  
        fato_valido.price,
        dim_imovel.room_type,
        dim_imovel.property_type,
        dim_imovel.accommodates,
        fato_valido.data_referencia,
    )
    .orderBy(col("price").desc())
)

print(f"Top 15 anúncios de maior preço em {bairros_investigar}:")
display(anuncios_extremos.limit(15))

# COMMAND ----------

# MAGIC %md
# MAGIC **Interpretação:** verifique se os valores mais altos pertencem sempre ao mesmo `sk_imovel`
# MAGIC repetido em vários meses ou se são imóveis distintos (mais compatível com
# MAGIC outliers legítimos, ainda que raros). 

# COMMAND ----------

pd_bairros = preco_por_bairro.limit(15).toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=pd_bairros, y="neighbourhood_cleansed", x="preco_mediano", color="steelblue")
plt.title("Top 15 bairros por preço mediano da diária (mín. 100 anúncios)")
plt.xlabel("Preço mediano (R$)")
plt.ylabel("Bairro")
plt.tight_layout()
plt.show()

# COMMAND ----------


# MAGIC %md
# MAGIC ## Pergunta 2 — Tipo de imóvel e capacidade influenciam mais que a localização?

# COMMAND ----------

preco_por_tipo_capacidade = (
    fato_valido
    .join(dim_imovel, fato_valido.sk_imovel == dim_imovel.sk_imovel, "inner")
    .groupBy("room_type", "accommodates")
    .agg(
        spark_round(avg("price"), 2).alias("preco_medio"),
        count("*").alias("qtd_anuncios"),
    )
    .filter(col("qtd_anuncios") >= 100)
    .orderBy(col("room_type"), col("accommodates"))
)

display(preco_por_tipo_capacidade)

# COMMAND ----------

pd_tipo_capacidade = preco_por_tipo_capacidade.filter(col("accommodates") <= 10).toPandas()

plt.figure(figsize=(10, 6))
sns.lineplot(data=pd_tipo_capacidade, x="accommodates", y="preco_medio", hue="room_type", marker="o")
plt.title("Preço médio por capacidade de hóspedes e tipo de acomodação")
plt.xlabel("Capacidade de hóspedes (accommodates)")
plt.ylabel("Preço médio (R$)")
plt.tight_layout()
plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Pergunta 3 — Existe correlação entre avaliação (review score) e preço?

# COMMAND ----------

fato_com_review = fato_valido.filter(col("review_scores_rating").isNotNull())

correlacao_review_preco = fato_com_review.select(
    corr("review_scores_rating", "price").alias("correlacao_rating_preco")
).collect()[0]["correlacao_rating_preco"]

print(f"Correlação entre review_scores_rating e price: {correlacao_review_preco:.4f}")
print("(valores próximos de 0 = correlação fraca/inexistente; próximos de 1 ou -1 = correlação forte)")

# COMMAND ----------

# Agrupando em faixas de nota para visualizar a relação de forma mais interpretável
from pyspark.sql.functions import when as spark_when

fato_com_faixa_nota = fato_com_review.withColumn(
    "faixa_nota",
    spark_when(col("review_scores_rating") >= 95, "95-100 (excelente)")
    .when(col("review_scores_rating") >= 90, "90-94 (muito bom)")
    .when(col("review_scores_rating") >= 80, "80-89 (bom)")
    .otherwise("< 80 (regular/ruim)")
)

preco_por_faixa_nota = (
    fato_com_faixa_nota
    .groupBy("faixa_nota")
    .agg(spark_round(avg("price"), 2).alias("preco_medio"), count("*").alias("qtd_anuncios"))
    .orderBy(col("preco_medio").desc())
)

display(preco_por_faixa_nota)

pd_faixa_nota = preco_por_faixa_nota.toPandas()
plt.figure(figsize=(8, 5))
sns.barplot(data=pd_faixa_nota, x="faixa_nota", y="preco_medio", color="darkorange")
plt.title("Preço médio por faixa de avaliação")
plt.xlabel("Faixa de avaliação (review_scores_rating)")
plt.ylabel("Preço médio (R$)")
plt.tight_layout()
plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Pergunta 4 — Os preços variam entre alta e baixa temporada?

# COMMAND ----------

preco_por_estacao = (
    fato_valido
    .join(dim_tempo, fato_valido.data_referencia == dim_tempo.data_referencia, "inner")
    .groupBy("estacao_do_ano")
    .agg(
        spark_round(avg("price"), 2).alias("preco_medio"),
        spark_round(median("price"), 2).alias("preco_mediano"),
        count("*").alias("qtd_anuncios"),
    )
)

# Ordem lógica das estações para o gráfico, em vez de ordem alfabética
ordem_estacoes = ["Verão", "Outono", "Inverno", "Primavera"]
pd_estacao = preco_por_estacao.toPandas()
pd_estacao["estacao_do_ano"] = pd_estacao["estacao_do_ano"].astype("category")
pd_estacao["estacao_do_ano"] = pd_estacao["estacao_do_ano"].cat.set_categories(ordem_estacoes)
pd_estacao = pd_estacao.sort_values("estacao_do_ano")

display(preco_por_estacao)

plt.figure(figsize=(8, 5))
sns.barplot(data=pd_estacao, x="estacao_do_ano", y="preco_medio", color="seagreen")
plt.title("Preço médio por estação do ano")
plt.xlabel("Estação do ano")
plt.ylabel("Preço médio (R$)")
plt.tight_layout()
plt.show()


# COMMAND ----------

from pyspark.sql.functions import date_format

preco_por_mes = (
    fato_valido
    .join(dim_tempo, fato_valido.data_referencia == dim_tempo.data_referencia, "inner")
    .groupBy(dim_tempo.data_referencia, "ano", "mes")
    .agg(spark_round(avg("price"), 2).alias("preco_medio"))
    .orderBy("data_referencia")
)

pd_mes = preco_por_mes.toPandas()
pd_mes["ano_mes"] = pd_mes["ano"].astype(str) + "-" + pd_mes["mes"].astype(str).str.zfill(2)

plt.figure(figsize=(14, 5))
sns.lineplot(data=pd_mes, x="ano_mes", y="preco_medio", marker="o")
plt.title("Preço médio da diária por mês (série completa)")
plt.xlabel("Mês de referência")
plt.ylabel("Preço médio (R$)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


# COMMAND ----------

# MAGIC %md
# MAGIC ## Pergunta 5 — Superhosts conseguem cobrar preços acima da média?

# COMMAND ----------

fato_com_superhost = fato_valido.filter(col("host_is_superhost").isNotNull())

preco_por_superhost = (
    fato_com_superhost
    .groupBy("host_is_superhost")
    .agg(
        spark_round(avg("price"), 2).alias("preco_medio"),
        spark_round(median("price"), 2).alias("preco_mediano"),
        count("*").alias("qtd_anuncios"),
    )
)

display(preco_por_superhost)


# COMMAND ----------

preco_superhost_por_tipo = (
    fato_com_superhost
    .join(dim_imovel, fato_com_superhost.sk_imovel == dim_imovel.sk_imovel, "inner")
    .groupBy("room_type", "host_is_superhost")
    .agg(
        spark_round(avg("price"), 2).alias("preco_medio"),
        count("*").alias("qtd_anuncios"),
    )
    .orderBy("room_type", "host_is_superhost")
)

display(preco_superhost_por_tipo)

pd_superhost_tipo = preco_superhost_por_tipo.toPandas()
plt.figure(figsize=(9, 5))
sns.barplot(data=pd_superhost_tipo, x="room_type", y="preco_medio", hue="host_is_superhost")
plt.title("Preço médio por tipo de imóvel, comparando superhost vs. não-superhost")
plt.xlabel("Tipo de acomodação")
plt.ylabel("Preço médio (R$)")
plt.legend(title="Superhost")
plt.tight_layout()
plt.show()

